# Этап 3б: четыре кандидата рецепта через ворота по живым вопросам

Продолжение [`kaggle_phase_d_recipe_commit.ipynb`](kaggle_phase_d_recipe_commit.ipynb).
Там рецепт подбирался по синтетическому dev, и из девяти этапов сохранены веса
только двух: `a-3ep` и `d-matryoshka`. Живые вопросы при подборе не смотрели
ни разу, а когда посмотрели, дообученная модель оказалась на них хуже
необученной на 0,062.

Здесь переобучаются четыре кандидата, названные до прогона
(`orchestration/EXPERIMENTS.md` части Б, раздел S12): `a-1ep`, `a-2ep`,
`c-kl`, `c-oba`. Веса нужны затем, чтобы прогнать каждого по всему набору
запросов и применить ворота. Правило применения ворот записано до того, как
увидено хоть одно живое число, и число применений идёт в отчёт.

**Журнал этапа отдельный, и это не придирка.** `Journal.add` сравнивает
новый этап с последним принятым, а в прежнем журнале последним стоит сетка
по эпохам. Допиши туда `a-1ep` сейчас, и его вердиктом станет сравнение
с `e-5ep`, а таблица рецепта переставится. Поэтому прежний журнал остаётся
как есть, а кандидаты считаются своей цепочкой от той же нулевой точки.

**Нулевая точка проверяется до обучения.** Образ Kaggle обновляется без
предупреждения, а таблица сравнивает этапы попарно: если между ними
отличается ещё и версия библиотеки, часть разницы окажется её. Шаг
`контроль` считает нулевую точку заново и падает, если она не воспроизвелась.
Четыре минуты против двух часов обучения, которое потом нельзя сравнивать.

**Входы:** датасеты `ru-fin-retrieval` и `ru-fin-queries`, модель
`intfloat/multilingual-e5-small`. Учитель и судья здесь не нужны: оценки
учителя посчитаны один раз и лежат в `train_prepared.jsonl`.

In [ ]:
# ──────── НАСТРОЙКИ ПРОГОНА ────────
# Четыре кандидата. Порядок обучения — порядок рецепта: так цепочка вердиктов
# читается как ответ на вопрос «что даёт эта составляющая», а не как
# случайная перестановка.
ТЕГИ_ОБУЧЕНИЕ = ['a-1ep', 'a-2ep', 'c-kl', 'c-oba']
# Порядок финала другой: матрицу эмбеддингов корпуса сохраняет первая метка,
# а нужна она тому кандидату, который пошёл бы в публикацию. По dev это
# `c-oba` (0,616 из девяти этапов), поэтому он и первый.
ТЕГИ_ФИНАЛ = ['c-oba', 'c-kl', 'a-2ep', 'a-1ep']
БАТЧ_ОЦЕНКИ = 128  # размер батча при подсчёте эмбеддингов, не при обучении

import time
НАЧАЛО = time.time()


def прошло() -> str:
    с = time.time() - НАЧАЛО
    return f'{с // 3600:.0f} ч {(с % 3600) // 60:02.0f} мин'


print('обучаем:', ', '.join(ТЕГИ_ОБУЧЕНИЕ))
print('финал по:', ', '.join(ТЕГИ_ФИНАЛ))

In [ ]:
import os
# Хаб запрещён, карта одна. Ставится до первого импорта torch
# и transformers, иначе библиотеки успеют прочитать настройки.
for ключ in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE'):
    os.environ[ключ] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print('офлайн и одна карта')

## 1. Карта и библиотеки

Версии печатаются, а не подгоняются: шаг `контроль` ниже проверит, что на
этом образе нулевая точка та же. Печать нужна затем, чтобы при расхождении
было видно, что именно поменялось.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

import torch, transformers, sentence_transformers
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('sentence-transformers', sentence_transformers.__version__)
print('карт видно:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'нет карты: Settings -> Accelerator'
# На Turing нет bf16, и обучение идёт в fp32 — это решение рецепта,
# а не умолчание библиотеки. Печатается для записи в отчёт.
print('bf16 поддерживается:', torch.cuda.is_bf16_supported())

from sentence_transformers import SentenceTransformerTrainer  # noqa: F401
from sentence_transformers.losses import (  # noqa: F401
    CachedMultipleNegativesRankingLoss, MatryoshkaLoss)
import datasets, accelerate
print('datasets', datasets.__version__, '| accelerate', accelerate.__version__)
print('всё, что нужно обучению, на месте')

## 2. Код, данные, подготовка и журнал

In [ ]:
import glob, gzip, json, os, shutil, zipfile


def find_source(root='/kaggle/input', max_depth=5):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            stale = ('__notebook__.ipynb' in files or '__output__.json' in files)
            found.append((1 if stale else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


src, archive = find_source()
assert src, 'датасет с кодом не подключён'
print('код:', src)
if archive:
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for имя in os.listdir(src):
        путь, цель = os.path.join(src, имя), os.path.join('/kaggle/working', имя)
        if os.path.isdir(путь):
            shutil.copytree(путь, цель, dirs_exist_ok=True)
        elif имя.endswith(('.py', '.md')):
            shutil.copyfile(путь, цель)

# Каноническая нарезка берётся готовой: границы фрагмента зависят от версии
# токенизатора, а идентификатор позиционный — эталон съехал бы молча.
dst = '/kaggle/working/chunks/base.jsonl'
if os.path.exists(dst):
    os.remove(dst)
найдено = (glob.glob('/kaggle/input/**/chunks_base.jsonl', recursive=True)
           + glob.glob('/kaggle/input/**/chunks_base.jsonl.gz', recursive=True))
assert найдено, 'нет chunks_base.jsonl: подключите датасет ru-fin-queries'
os.makedirs('/kaggle/working/chunks', exist_ok=True)
opener = gzip.open if найдено[0].endswith('.gz') else open
with opener(найдено[0], 'rb') as fi, open(dst, 'wb') as fo:
    shutil.copyfileobj(fi, fo, length=1 << 20)
n = sum(1 for _ in open(dst, encoding='utf-8'))
assert n == 62594, f'ожидалось 62594, а не {n}: нарезка не каноническая'
print('нарезка:', n, 'фрагментов')

# Подготовка D0: трудные негативы и оценки учителя. Без неё этапы
# с негативами и дистилляцией учиться не на чем.
подготовка = glob.glob('/kaggle/input/**/train_prepared.jsonl', recursive=True)
assert подготовка, 'нет train_prepared.jsonl: подключите датасет ru-fin-queries'
print('подготовка:', подготовка[0],
      sum(1 for _ in open(подготовка[0], encoding='utf-8')), 'записей')

In [ ]:
# Журнал этапа 3б: из прежнего берётся только нулевая точка.
# Дописывать кандидатов в прежний журнал нельзя — `Journal.add` сравнивает
# каждого с последним принятым, а последней там стоит сетка по эпохам:
# вердиктом `a-1ep` оказалось бы сравнение с `e-5ep`, и таблица рецепта
# переставилась бы. Прежние цифры остаются как есть и сравниваются с новыми
# после прогона, вручную: это ещё одна проверка воспроизводимости.
os.makedirs('/kaggle/working/train', exist_ok=True)
прежний = glob.glob('/kaggle/input/**/journal.json', recursive=True)
assert прежний, 'нет journal.json: подключите датасет ru-fin-queries'
было = json.load(open(прежний[0], encoding='utf-8'))
база = [э for э in было['этапы'] if э['tag'] == 'базовая']
assert база, 'в прежнем журнале нет нулевой точки, проверять нечего'
ЖУРНАЛ = '/kaggle/working/train/journal_3b.json'
with open(ЖУРНАЛ, 'w', encoding='utf-8') as f:
    json.dump({'правило приёмки': было['правило приёмки'], 'этапы': база},
              f, ensure_ascii=False, indent=1)

print('в прежнем журнале:', [э['tag'] for э in было['этапы']])
print('прежние dev NDCG@10:',
      {э['tag']: round(э['dev']['NDCG@10']['mean'], 4) for э in было['этапы']})
print(f"журнал этапа 3б: только нулевая точка, "
      f"{база[0]['dev']['NDCG@10']['mean']:.5f}")
print('\nпрошло', прошло())

## 3. Контроль, обучение, финал

Три шага одним процессом каждый, по порядку. `контроль` считает нулевую точку
заново и падает, если она не воспроизвелась: тогда обучать нет смысла, потому
что сравнивать результат будет не с чем. `train` обучает четырёх кандидатов
и после каждого считает dev. `final` прогоняет каждого по всему набору
запросов и сохраняет выдачи, по которым на маке применяются ворота.

Веса здесь не убираются по ходу (`--keep-only-best` не передаётся): финалу
нужны все четыре, и обучить их заново стоило бы двух часов карты.

In [ ]:
import subprocess, sys, time

ШАГИ = [('контроль', []), ('train', ТЕГИ_ОБУЧЕНИЕ), ('final', ТЕГИ_ФИНАЛ)]
КОД = 0
ВРЕМЯ = {}
for шаг, теги in ШАГИ:
    cmd = [sys.executable, 'run_phase_d.py', '--step', шаг,
           '--journal', ЖУРНАЛ, '--prepared', подготовка[0],
           '--batch-size', str(БАТЧ_ОЦЕНКИ)]
    if теги:
        cmd += ['--tags'] + list(теги)
    print(f'\n{"=" * 70}\n==== {шаг}: {" ".join(cmd)}\n{"=" * 70}', flush=True)
    t = time.time()
    p = subprocess.run(cmd, cwd='/kaggle/working')
    ВРЕМЯ[шаг] = round((time.time() - t) / 60, 1)
    print(f'\n{шаг}: код возврата {p.returncode}, {ВРЕМЯ[шаг]} мин, '
          f'всего прошло {прошло()}', flush=True)
    if p.returncode:
        КОД = p.returncode
        print(f'шаг «{шаг}» упал, дальше не идём', flush=True)
        break
print('\nминут по шагам:', ВРЕМЯ)

## 4. Таблица и уборка

In [ ]:
import os, shutil, sys
sys.path.insert(0, '/kaggle/working')
from rufin.training.journal import Journal

if os.path.exists(ЖУРНАЛ):
    j = Journal.load(ЖУРНАЛ)
    print(j.table())
    print('\nэтапов в журнале этапа 3б:', len(j.entries))
    print('прежние dev для сверки:',
          {э['tag']: round(э['dev']['NDCG@10']['mean'], 4) for э in было['этапы']
           if э['tag'] in ТЕГИ_ОБУЧЕНИЕ})
else:
    print('журнала нет: ни один шаг не досчитался')

for путь in ('/kaggle/working/runs', '/kaggle/working/embeddings'):
    for база, _, файлы in os.walk(путь):
        for f in sorted(файлы):
            п = os.path.join(база, f)
            print(f'   {os.path.relpath(п, "/kaggle/working"):<48} '
                  f'{os.path.getsize(п) // 1024:>8} КБ')

# Нарезка и корпус приехали из входов, а рабочая папка целиком становится
# выводом версии: возить четверть гигабайта обратно незачем.
shutil.rmtree('/kaggle/working/chunks', ignore_errors=True)
for имя in ('acts.jsonl', 'acts.jsonl.gz'):
    if os.path.exists(os.path.join('/kaggle/working', имя)):
        os.remove(os.path.join('/kaggle/working', имя))

# Веса остаются только у кандидата, который пошёл бы в публикацию. Можно
# потому, что каждый этап обучается с исходных весов: для продолжения нужен
# только журнал. Убираются лишь когда финал досчитался — иначе перезапуск
# финала обучал бы заново.
веса = '/kaggle/working/weights'
if КОД == 0 and os.path.isdir(веса):
    for имя in sorted(os.listdir(веса)):
        if имя != ТЕГИ_ФИНАЛ[0] and os.path.isdir(os.path.join(веса, имя)):
            shutil.rmtree(os.path.join(веса, имя), ignore_errors=True)
            print(f'веса {имя} убраны: цифры в журнале, обучение всё равно '
                  f'начинается с исходных')

размер = sum(os.path.getsize(os.path.join(б, и))
             for б, _, ф in os.walk('/kaggle/working') for и in ф)
print(f'\nвывод версии: {размер / 1073741824:.2f} ГБ, прошло {прошло()}')

# Упавший шаг обязан красить тетрадь в красное. Иначе Kaggle пишет
# «Successfully ran» на несделанной работе, и это видно только в журнале.
assert КОД == 0, f'прогон вернул {КОД}, смотрите вывод выше